In [0]:
%run ../Notebooks/00_Configuration

Configuration Loaded Successfully


In [0]:
%run ../framework/01_Utility_Functions

Configuration Loaded Successfully


In [0]:
dbutils.widgets.text("table_name", "Customer")
dbutils.widgets.text("pipeline_run_id", "")

table_name = dbutils.widgets.get("table_name")
pipeline_run_id = dbutils.widgets.get("pipeline_run_id")

In [0]:
from pyspark.sql.functions import *

metadata_df = spark.table("insurance_metadata.bronze_config")

In [0]:
from pyspark.sql.functions import col

# dbutils.widgets.text("table_name", "Customer")

# table_name = dbutils.widgets.get("table_name")

metadata_row = (
    metadata_df
    .filter(
        (col("table_name") == table_name) &
        (col("scd_enabled") == "Y")
    )
    .first()
)

primary_key = metadata_row["primary_key"]

gold_table = metadata_row["gold_table"]

history_table = metadata_row["history_table"]

compare_columns = [
    c.strip()
    for c in metadata_row["compare_columns"].split(",")
]

print("Primary Key      :", primary_key)
print("Gold Table       :", gold_table)
print("History Table    :", history_table)
print("Compare Columns  :", compare_columns)

Primary Key      : customer_id
Gold Table       : dim_customer
History Table    : dim_customer_history
Compare Columns  : ['first_name', 'last_name', 'email', 'phone', 'country', 'city', 'gender']


In [0]:
gold_table_name = f"{gold_schema}.{gold_table}"

gold_df = spark.table(gold_table_name)

print(f"Gold Table : {gold_table_name}")
print(f"Rows : {gold_df.count()}")

display(gold_df.limit(5))

Gold Table : insurance_gold.dim_customer
Rows : 911


customer_id,first_name,last_name,email,phone,country,city,registration_date,date_of_birth,gender,ingestion_timestamp,source_file_name,load_id,pipeline_run_id,source_system
1001,Bruis,Myall,bmyall0@hibu.com,3604158081,United States,Vancouver,2022-09-23 11:10:45,2023-02-13 03:14:05,Male,2026-07-17T10:48:45.018931Z,abfss://landing@adlsinsurancedevstorage.dfs.core.windows.net/customer/Customer-acMVLthHdA.csv,4d4a6739-9bdd-4754-9041-534fca1b7abf,7f1b0d87-beb8-4e10-b762-023fe277a227,Customer_CSV
1002,Ellene,MacSporran,emacsporran1@mapquest.com,9011721346,United States,Memphis,2022-10-15 18:01:12,2023-03-28 13:09:21,Female,2026-07-17T10:48:45.018931Z,abfss://landing@adlsinsurancedevstorage.dfs.core.windows.net/customer/Customer-acMVLthHdA.csv,7ac2efdd-665f-4eec-8f4e-39f94d7b82a5,a2571a77-8746-493e-9d17-43ce1f27798b,Customer_CSV
1003,Isabelita,Powlesland,ipowlesland2@chicagotribune.com,7081890902,United Kingdom,Liverpool,2022-06-20 03:55:09,2022-09-08 06:25:57,Female,2026-07-17T10:48:45.018931Z,abfss://landing@adlsinsurancedevstorage.dfs.core.windows.net/customer/Customer-acMVLthHdA.csv,72e707fb-35b8-4c57-9478-8c2000b01d7f,bdcb3d51-fa9b-411e-a4b7-a3967a787eeb,Customer_CSV
1004,Elden,Tonn,etonn3@t-online.de,8156364393,United States,Rockford,2023-03-08 09:56:41,2023-01-07 01:31:46,Male,2026-07-17T10:48:45.018931Z,abfss://landing@adlsinsurancedevstorage.dfs.core.windows.net/customer/Customer-acMVLthHdA.csv,2340d618-aa6b-40eb-8df3-bb2f899016ed,f66aa66b-fdcb-4cdb-bcf9-d902373e7893,Customer_CSV
1005,Edd,null,ediviny4@list-manage.com,4053842760,United States,null,2022-08-14 19:04:28,2022-04-23 15:11:46,Male,2026-07-17T10:48:45.018931Z,abfss://landing@adlsinsurancedevstorage.dfs.core.windows.net/customer/Customer-acMVLthHdA.csv,2efecdf8-b707-4d53-8291-bafb1226cc91,27df8248-b7a6-410b-b95b-7ae79521f65c,Customer_CSV


In [0]:
history_table_name = f"{gold_schema}.{history_table}"

from pyspark.sql.utils import AnalysisException

try:

    history_df = spark.table(history_table_name)

    print("History table exists")

    print("Rows :", history_df.count())

except AnalysisException:

    history_df = None

    print("History table does not exist.")

History table exists
Rows : 911


In [0]:
# display(
#     spark.table(history_table_name).limit(10)
# )

In [0]:
from pyspark.sql.functions import current_timestamp, lit

if not spark.catalog.tableExists(history_table_name):

    (
        gold_df
        .withColumn(
            "effective_start_date",
            current_timestamp()
        )
        .withColumn(
            "effective_end_date",
            lit(None).cast("timestamp")
        )
        .withColumn(
            "is_current",
            lit(True)
        )
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(history_table_name)
    )

    print("History table created.")

else:

    print("History table already exists.")

# (
#     gold_df
#     .withColumn("effective_start_date", current_timestamp())
#     .withColumn("effective_end_date", lit(None).cast("timestamp"))
#     .withColumn("is_current", lit(True))
#     .write
#     .format("delta")
#     .mode("overwrite")
#     .saveAsTable(history_table_name)
# )

# history_df = spark.table(history_table_name)

# print("History Reset Successfully")
# print("Rows :", history_df.count())

History table already exists.


In [0]:
current_history_df = (
    history_df
    .filter(col("is_current") == True)
)

print("Current History Rows :", current_history_df.count())

Current History Rows : 911


In [0]:
from pyspark.sql.functions import col

# Rename every history column except the primary key
history_columns = []

for column in current_history_df.columns:

    if column == primary_key:
        history_columns.append(col(column))

    else:
        history_columns.append(
            col(column).alias(f"old_{column}")
        )

history_renamed_df = current_history_df.select(*history_columns)

comparison_df = (
    gold_df.alias("new")
    .join(
        history_renamed_df.alias("old"),
        on=primary_key,
        how="left"
    )
)

print("Comparison Rows :", comparison_df.count())

display(comparison_df.limit(5))

print("Comparison Rows :", comparison_df.count())

Comparison Rows : 911


customer_id,first_name,last_name,email,phone,country,city,registration_date,date_of_birth,gender,ingestion_timestamp,source_file_name,load_id,pipeline_run_id,source_system,old_first_name,old_last_name,old_email,old_phone,old_country,old_city,old_registration_date,old_date_of_birth,old_gender,old_ingestion_timestamp,old_source_file_name,old_load_id,old_pipeline_run_id,old_source_system,old_effective_start_date,old_effective_end_date,old_is_current
1004,Elden,Tonn,etonn3@t-online.de,8156364393,United States,Rockford,2023-03-08 09:56:41,2023-01-07 01:31:46,Male,2026-07-17T10:48:45.018931Z,abfss://landing@adlsinsurancedevstorage.dfs.core.windows.net/customer/Customer-acMVLthHdA.csv,2340d618-aa6b-40eb-8df3-bb2f899016ed,f66aa66b-fdcb-4cdb-bcf9-d902373e7893,Customer_CSV,Elden,Tonn,etonn3@t-online.de,8156364393,United States,Rockford,2023-03-08 09:56:41,2023-01-07 01:31:46,Male,2026-07-17T10:48:45.018931Z,abfss://landing@adlsinsurancedevstorage.dfs.core.windows.net/customer/Customer-acMVLthHdA.csv,2340d618-aa6b-40eb-8df3-bb2f899016ed,f66aa66b-fdcb-4cdb-bcf9-d902373e7893,Customer_CSV,2026-07-19T16:14:22.347802Z,null,true
1005,Edd,null,ediviny4@list-manage.com,4053842760,United States,null,2022-08-14 19:04:28,2022-04-23 15:11:46,Male,2026-07-17T10:48:45.018931Z,abfss://landing@adlsinsurancedevstorage.dfs.core.windows.net/customer/Customer-acMVLthHdA.csv,2efecdf8-b707-4d53-8291-bafb1226cc91,27df8248-b7a6-410b-b95b-7ae79521f65c,Customer_CSV,Edd,null,ediviny4@list-manage.com,4053842760,United States,null,2022-08-14 19:04:28,2022-04-23 15:11:46,Male,2026-07-17T10:48:45.018931Z,abfss://landing@adlsinsurancedevstorage.dfs.core.windows.net/customer/Customer-acMVLthHdA.csv,2efecdf8-b707-4d53-8291-bafb1226cc91,27df8248-b7a6-410b-b95b-7ae79521f65c,Customer_CSV,2026-07-19T16:14:22.347802Z,null,true
1003,Isabelita,Powlesland,ipowlesland2@chicagotribune.com,7081890902,United Kingdom,Liverpool,2022-06-20 03:55:09,2022-09-08 06:25:57,Female,2026-07-17T10:48:45.018931Z,abfss://landing@adlsinsurancedevstorage.dfs.core.windows.net/customer/Customer-acMVLthHdA.csv,72e707fb-35b8-4c57-9478-8c2000b01d7f,bdcb3d51-fa9b-411e-a4b7-a3967a787eeb,Customer_CSV,Isabelita,Powlesland,ipowlesland2@chicagotribune.com,7081890902,United Kingdom,Liverpool,2022-06-20 03:55:09,2022-09-08 06:25:57,Female,2026-07-17T10:48:45.018931Z,abfss://landing@adlsinsurancedevstorage.dfs.core.windows.net/customer/Customer-acMVLthHdA.csv,72e707fb-35b8-4c57-9478-8c2000b01d7f,bdcb3d51-fa9b-411e-a4b7-a3967a787eeb,Customer_CSV,2026-07-19T16:14:22.347802Z,null,true
1002,Ellene,MacSporran,emacsporran1@mapquest.com,9011721346,United States,Memphis,2022-10-15 18:01:12,2023-03-28 13:09:21,Female,2026-07-17T10:48:45.018931Z,abfss://landing@adlsinsurancedevstorage.dfs.core.windows.net/customer/Customer-acMVLthHdA.csv,7ac2efdd-665f-4eec-8f4e-39f94d7b82a5,a2571a77-8746-493e-9d17-43ce1f27798b,Customer_CSV,Ellene,MacSporran,emacsporran1@mapquest.com,9011721346,United States,Memphis,2022-10-15 18:01:12,2023-03-28 13:09:21,Female,2026-07-17T10:48:45.018931Z,abfss://landing@adlsinsurancedevstorage.dfs.core.windows.net/customer/Customer-acMVLthHdA.csv,7ac2efdd-665f-4eec-8f4e-39f94d7b82a5,a2571a77-8746-493e-9d17-43ce1f27798b,Customer_CSV,2026-07-19T16:14:22.347802Z,null,true
1001,Bruis,Myall,bmyall0@hibu.com,3604158081,United States,Vancouver,2022-09-23 11:10:45,2023-02-13 03:14:05,Male,2026-07-17T10:48:45.018931Z,abfss://landing@adlsinsurancedevstorage.dfs.core.windows.net/customer/Customer-acMVLthHdA.csv,4d4a6739-9bdd-4754-9041-534fca1b7abf,7f1b0d87-beb8-4e10-b762-023fe277a227,Customer_CSV,Bruis,Myall,bmyall0@hibu.com,3604158081,United States,Vancouver,2022-09-23 11:10:45,2023-02-13 03:14:05,Male,2026-07-17T10:48:45.018931Z,abfss://landing@adlsinsurancedevstorage.dfs.core.windows.net/customer/Customer-acMVLthHdA.csv,4d4a6739-9bdd-4754-9041-534fca1b7abf,7f1b0d87-beb8-4e10-b762-023fe277a227,Customer_CSV,2026-07-19T16:14:22.347802Z,null,true


Comparison Rows : 911


In [0]:
new_records = comparison_df.filter(
    col("old.old_effective_start_date").isNull()
)

new_record_count = new_records.count()

print("New Records :", new_record_count )

display(new_records.limit(5))

New Records : 0


customer_id,first_name,last_name,email,phone,country,city,registration_date,date_of_birth,gender,ingestion_timestamp,source_file_name,load_id,pipeline_run_id,source_system,old_first_name,old_last_name,old_email,old_phone,old_country,old_city,old_registration_date,old_date_of_birth,old_gender,old_ingestion_timestamp,old_source_file_name,old_load_id,old_pipeline_run_id,old_source_system,old_effective_start_date,old_effective_end_date,old_is_current


In [0]:
# from functools import reduce
# from pyspark.sql.functions import col

# --------------------------------------------------
# Compare only business columns
# --------------------------------------------------

# comparison_conditions = [

#     ~col(f"new.{column}").eqNullSafe(
#         col(f"old.old_{column}")
#     )

#     for column in compare_columns

# ]

# change_condition = reduce(
#     lambda x, y: x | y,
#     comparison_conditions
# )

# --------------------------------------------------
# IMPORTANT:
# Only existing customers can be considered "changed"
# New customers must not enter this dataframe.
# --------------------------------------------------

# changed_records = (

#     comparison_df

#         .filter(
#             col("old.old_effective_start_date").isNotNull()
#         )

#         .filter(change_condition)

# )

# changed_record_count = changed_records.count()

# print("=" * 60)
# print("Changed Records :", changed_record_count)
# print("=" * 60)

# display(changed_records.limit(10))


from functools import reduce

comparison_conditions = [

    ~col(f"new.{column}").eqNullSafe(
        col(f"old.old_{column}")
    )

    for column in compare_columns

]

change_condition = reduce(
    lambda x, y: x | y,
    comparison_conditions
)

changed_records = comparison_df.filter(
    col("old.old_effective_start_date").isNotNull()
).filter(
    change_condition
)

changed_record_count = changed_records.count()

print("Changed Records :", changed_record_count)

display(changed_records.limit(10))

Changed Records : 0


customer_id,first_name,last_name,email,phone,country,city,registration_date,date_of_birth,gender,ingestion_timestamp,source_file_name,load_id,pipeline_run_id,source_system,old_first_name,old_last_name,old_email,old_phone,old_country,old_city,old_registration_date,old_date_of_birth,old_gender,old_ingestion_timestamp,old_source_file_name,old_load_id,old_pipeline_run_id,old_source_system,old_effective_start_date,old_effective_end_date,old_is_current


In [0]:
display(
    changed_records.select(
        col(f"new.{primary_key}").alias(primary_key),
        *[
            col(f"new.{c}").alias(c)
            for c in compare_columns
        ]
    ).limit(10)
)

customer_id,first_name,last_name,email,phone,country,city,gender


In [0]:
print("Current History Rows :", current_history_df.count())
print("Comparison Rows :", comparison_df.count())
print("New Records :", new_record_count)
print("Changed Records :", changed_record_count)

Current History Rows : 911
Comparison Rows : 911
New Records : 0
Changed Records : 0


In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import current_timestamp

history_delta = DeltaTable.forName(
    spark,
    history_table_name
)

changed_keys = (
    changed_records
    .select(primary_key)
    .distinct()
)

(
    history_delta.alias("hist")
    .merge(
        changed_keys.alias("chg"),
        f"hist.{primary_key} = chg.{primary_key} "
        "AND hist.is_current = true"
    )
    .whenMatchedUpdate(
        set={
            "is_current": "false",
            "effective_end_date": "current_timestamp()"
        }
    )
    .execute()
)

print("Existing versions expired successfully.")

Existing versions expired successfully.


In [0]:
from pyspark.sql.functions import current_timestamp, lit

# -------------------------------
# New Business Keys
# -------------------------------

new_insert_df = (
    new_records
    .select("new.*")
)

# # -------------------------------
# # Changed Business Keys
# # -------------------------------

changed_insert_df = (
    changed_records
    .select("new.*")
)


# final_insert_df = (

#     new_insert_df

#     .unionByName(changed_insert_df)

#     .dropDuplicates([primary_key])

#     .withColumn(
#         "effective_start_date",
#         current_timestamp()
#     )

#     .withColumn(
#         "effective_end_date",
#         lit(None).cast("timestamp")
#     )

#     .withColumn(
#         "is_current",
#         lit(True)
#     )

# )

# -------------------------------
# Combine Both
# -------------------------------

final_insert_df = (
    new_insert_df
    .unionByName(changed_insert_df)
    .dropDuplicates([primary_key])
    .withColumn("effective_start_date", current_timestamp())
    .withColumn("effective_end_date", lit(None).cast("timestamp"))
    .withColumn("is_current", lit(True))
)

insert_count = final_insert_df.count()


print("Rows to Insert :", insert_count)

display(final_insert_df.limit(10))

Rows to Insert : 0


customer_id,first_name,last_name,email,phone,country,city,registration_date,date_of_birth,gender,ingestion_timestamp,source_file_name,load_id,pipeline_run_id,source_system,effective_start_date,effective_end_date,is_current


In [0]:
if insert_count > 0:

    (
        final_insert_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(history_table_name)
    )

    print("Inserted", insert_count, "new SCD2 records.")

else:

    print("No new records to insert.")

No new records to insert.


In [0]:
pipeline_name = "Generic_SCD2"

rows_written = new_record_count + changed_record_count
gold_rows = gold_df.count()

write_audit(
    pipeline_name=pipeline_name,
    table_name=history_table,
    load_type="SCD2",
    rows_read = gold_rows,
    rows_written=rows_written,
    status="SUCCESS",
    error_message="",
    pipeline_run_id=pipeline_run_id
)

In [0]:
history_df = spark.table(history_table_name)

print("=" * 60)

print("Generic SCD2 Completed Successfully")

print("=" * 60)

print("Gold Rows       :", gold_df.count())

print("History Rows    :", history_df.count())

print("Current Records :", history_df.filter(col("is_current")==True).count())

print("=" * 60)

Generic SCD2 Completed Successfully
Gold Rows       : 911
History Rows    : 911
Current Records : 911
